## Tools
 Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [9]:
import os
from langchain_groq import ChatGroq 

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm= ChatGroq(
    model='openai/gpt-oss-120b',
    api_key=os.getenv("GROQ_API_KEY")
)

response = llm.invoke("What is the capital of France?")
response.content

'The capital of France is **Paris**.'

In [ ]:
from langchain.tools import tool

# decorator to mark a function as a tool that can be used by the agent
@tool

# this is a pythonic way of writing that the input is string implicitly defined
    
def get_weather(location: str) -> str:
    """Get the weather location"""

    return f"It's sunny in {location}!"


# to bind the tools with llm, we can use the bind_tools method. This will allow the agent to use the tools when generating responses.
model_with_tools = llm.bind_tools([get_weather])

In [16]:
## to invoke the tool, we can use the invoke method. This will allow the agent to use the tools when generating responses.

response = model_with_tools.invoke("What is the weather in New York?")
print(response.tool_calls)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
    print(f"id: {tool_call['id']}")

[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_5b3be4cf-14e3-40ac-9204-a12018d34813', 'type': 'tool_call'}]
Tool: get_weather
Arguments: {'location': 'New York'}
id: fc_5b3be4cf-14e3-40ac-9204-a12018d34813


## Tool Execution Loop

In [17]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "what is the weather in Boston?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Tool calls are executed and results are returned to the model
for tool_call in ai_message.tool_calls:
    # Execute the tool with the provided arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response generation
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny.


In [18]:
messages

[{'role': 'user', 'content': 'what is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_c2e39af5-fe08-4c96-b6af-7400296b6e3a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 125, 'total_tokens': 164, 'completion_time': 0.082028581, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.005172257, 'prompt_tokens_details': None, 'queue_time': 0.263664962, 'total_time': 0.087200838}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_84bf0227ec', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9dfc-8926-70d1-b341-37cae3ca2605-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_c2e39af5-fe08-4c96-b6af-7400296b6e3a